# NN Architecture 2A: Correlator + Deep Neural Network

**Reference**: Braca et al. (2022) - "Statistical Hypothesis Testing Based on Machine Learning: Large Deviations Analysis" + Neyman-Pearson Lemma

**Approach**: Combine optimal signal detection theory (matched filter/correlator) with discriminative deep learning

**Architecture**:
```
Input: [τ_correlator, h_estimate, SNR_local, energy] (4D features)
    ↓
Dense(256, ReLU, BatchNorm) → Dropout(0.3)
    ↓
Dense(128, ReLU, BatchNorm) → Dropout(0.3)
    ↓
Dense(64, ReLU) → Dropout(0.2)
    ↓
Dense(1, Sigmoid) → Binary output (P(authentic))
```

**Loss**: Binary Crossentropy with class weights (to handle imbalance)

**Optimization**: Adam, early stopping on validation FNR

In [ ]:
# ==============================================================================
# 1. IMPORTS & CONFIGURATION
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
import h5py
import os
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
import tensorflow.keras.backend as K

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# ==============================================================================
# 2. LOAD DATASET
# ==============================================================================

dataset_path = "dataset_nn_100k.h5"

if not os.path.exists(dataset_path):
    print(f"ERROR: {dataset_path} not found!")
    print("Please run NN_01_DataGeneration.ipynb first.")
else:
    with h5py.File(dataset_path, 'r') as f:
        X_train = f['X_train'][:]
        y_train = f['y_train'][:]
        X_val = f['X_val'][:]
        y_val = f['y_val'][:]
        X_test = f['X_test'][:]
        y_test = f['y_test'][:]
    
    print(f"✓ Dataset loaded successfully!")
    print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"  X_val:   {X_val.shape}, y_val: {y_val.shape}")
    print(f"  X_test:  {X_test.shape}, y_test: {y_test.shape}")
    print(f"  Train labels: {np.bincount(y_train)}")
    print(f"  Val labels:   {np.bincount(y_val)}")
    print(f"  Test labels:  {np.bincount(y_test)}")

In [ ]:
# ==============================================================================
# 3. BUILD DNN ARCHITECTURE (Reference: Braca et al. 2022)
# ==============================================================================

def build_dnn_correlator(input_dim=4, dropout_rates=[0.3, 0.3, 0.2]):
    """Build DNN Correlator architecture per Braca et al. 2022
    
    Args:
        input_dim: Number of input features (default 4)
        dropout_rates: Dropout rates for each layer
    
    Returns:
        model: Keras Sequential model
    """
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        
        # Layer 1
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rates[0]),
        
        # Layer 2
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rates[1]),
        
        # Layer 3
        layers.Dense(64, activation='relu'),
        layers.Dropout(dropout_rates[2]),
        
        # Output layer (binary classification)
        layers.Dense(1, activation='sigmoid')
    ])
    
    return model

# Build model
model = build_dnn_correlator(input_dim=X_train.shape[1])

# Compile with class weights to handle imbalance
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=BinaryCrossentropy(),
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

print("✓ Model architecture:")
model.summary()

In [ ]:
# ==============================================================================
# 4. TRAINING CALLBACKS
# ==============================================================================

# Calculate class weights to handle imbalance
class_weight = {
    0: np.sum(y_train == 1) / np.sum(y_train == 0),  # Upweight minority class
    1: 1.0
}

print(f"Class weights: {class_weight}")

# Early stopping based on validation loss
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Reduce learning rate on plateau
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Model checkpoint
model_checkpoint = callbacks.ModelCheckpoint(
    'model_dnn_correlator.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=0
)

print("✓ Callbacks configured")

In [ ]:
# ==============================================================================
# 5. TRAIN MODEL
# ==============================================================================

print("Training DNN Correlator model...")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=256,
    class_weight=class_weight,
    callbacks=[early_stopping, reduce_lr, model_checkpoint],
    verbose=1
)

print("\n✓ Training completed!")
print(f"  Final train loss: {history.history['loss'][-1]:.4f}")
print(f"  Final val loss:   {history.history['val_loss'][-1]:.4f}")
print(f"  Best epoch:       {len(history.history['loss']) - 10}")

In [ ]:
# ==============================================================================
# 6. EVALUATION METRICS
# ==============================================================================

# Make predictions
y_train_pred_proba = model.predict(X_train, verbose=0)
y_val_pred_proba = model.predict(X_val, verbose=0)
y_test_pred_proba = model.predict(X_test, verbose=0)

# Get binary predictions (threshold = 0.5)
y_train_pred = (y_train_pred_proba > 0.5).astype(int).flatten()
y_val_pred = (y_val_pred_proba > 0.5).astype(int).flatten()
y_test_pred = (y_test_pred_proba > 0.5).astype(int).flatten()

def calculate_metrics(y_true, y_pred, y_pred_proba, dataset_name="Test"):
    """Calculate comprehensive metrics"""
    
    # Binary classification metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_pred_proba)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # False Negative Rate (main metric)
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    
    # False Positive Rate
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    print(f"\n{dataset_name} Set Metrics:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f} (detects authentic)")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  AUC:       {auc:.4f}")
    print(f"  FNR:       {fnr:.6f} (false negatives)")
    print(f"  FPR:       {fpr:.6f} (false positives)")
    print(f"  Confusion: TP={tp}, FP={fp}, TN={tn}, FN={fn}")
    
    return {
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
        'auc': auc, 'fnr': fnr, 'fpr': fpr, 'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn
    }

# Calculate metrics for all sets
metrics_train = calculate_metrics(y_train, y_train_pred, y_train_pred_proba, "Train")
metrics_val = calculate_metrics(y_val, y_val_pred, y_val_pred_proba, "Validation")
metrics_test = calculate_metrics(y_test, y_test_pred, y_test_pred_proba, "Test")

In [ ]:
# ==============================================================================
# 7. VISUALIZATIONS
# ==============================================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Plot 1: Training history - Loss
axes[0, 0].plot(history.history['loss'], label='Train', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Model Loss (DNN Correlator)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Training history - Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Model Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Training history - AUC
axes[0, 2].plot(history.history['auc'], label='Train', linewidth=2)
axes[0, 2].plot(history.history['val_auc'], label='Validation', linewidth=2)
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('AUC')
axes[0, 2].set_title('Model AUC')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Confusion Matrix (Test)
cm_test = confusion_matrix(y_test, y_test_pred)
im = axes[1, 0].imshow(cm_test, cmap='Blues', interpolation='nearest')
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('True')
axes[1, 0].set_title('Confusion Matrix (Test Set)')
axes[1, 0].set_xticks([0, 1])
axes[1, 0].set_yticks([0, 1])
axes[1, 0].set_xticklabels(['Fraudulent', 'Authentic'])
axes[1, 0].set_yticklabels(['Fraudulent', 'Authentic'])
for i in range(2):
    for j in range(2):
        axes[1, 0].text(j, i, str(cm_test[i, j]), ha='center', va='center', color='white', fontsize=14)

# Plot 5: ROC Curve (Test)
fpr_test, tpr_test, thresholds_test = roc_curve(y_test, y_test_pred_proba)
axes[1, 1].plot(fpr_test, tpr_test, linewidth=2, label=f'AUC={metrics_test["auc"]:.4f}')
axes[1, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate')
axes[1, 1].set_title('ROC Curve (Test Set)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Plot 6: Prediction probability distribution
axes[1, 2].hist(y_test_pred_proba[y_test==0], alpha=0.6, label='Fraudulent (true)', bins=40)
axes[1, 2].hist(y_test_pred_proba[y_test==1], alpha=0.6, label='Authentic (true)', bins=40)
axes[1, 2].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Decision threshold')
axes[1, 2].set_xlabel('Predicted Probability')
axes[1, 2].set_ylabel('Count')
axes[1, 2].set_title('Prediction Probability Distribution (Test)')
axes[1, 2].legend()

plt.tight_layout()
plt.savefig('results_dnn_correlator.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Results visualization saved to 'results_dnn_correlator.png'")

In [ ]:
# ==============================================================================
# 8. SAVE RESULTS
# ==============================================================================

# Save model
model.save('model_dnn_correlator_final.h5')
print("✓ Model saved to 'model_dnn_correlator_final.h5'")

# Save metrics
results = {
    'train': metrics_train,
    'val': metrics_val,
    'test': metrics_test,
    'history': history.history
}

import json
with open('metrics_dnn_correlator.json', 'w') as f:
    # Convert numpy types to native Python types for JSON serialization
    results_json = {}
    for split in ['train', 'val', 'test']:
        results_json[split] = {k: float(v) if isinstance(v, (np.ndarray, np.integer)) else v 
                               for k, v in results[split].items()}
    results_json['history'] = {k: [float(v) for v in results['history'][k]] 
                               for k in results['history'].keys()}
    json.dump(results_json, f, indent=2)

print("✓ Metrics saved to 'metrics_dnn_correlator.json'")

# Summary table
print("\n" + "="*70)
print("SUMMARY: DNN CORRELATOR ARCHITECTURE (Braca et al. 2022)")
print("="*70)
print(f"\n{'Metric':<20} {'Train':<15} {'Val':<15} {'Test':<15}")
print("-"*70)
for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc', 'fnr', 'fpr']:
    print(f"{metric:<20} {metrics_train[metric]:<15.6f} {metrics_val[metric]:<15.6f} {metrics_test[metric]:<15.6f}")
print("="*70)

## References & Notes

### Architecture Justification

This DNN Correlator combines two fundamental theories:

1. **Neyman-Pearson Lemma** (Classical Signal Detection)
   - Optimal detector uses likelihood ratio
   - Matched filter (correlator) is optimal for Gaussian noise
   - Correlator output: $\tau = \sum_{i=1}^{L} y_i \cdot t_{\text{ref},i}$

2. **Large Deviations Theory** (Braca et al. 2022)
   - Exact asymptotic error probability: $P_e \propto \exp(-I(\theta))$
   - $I(\theta)$ is the Rate Function (Fenchel-Legendre transform)
   - ML can learn the decision boundary that approximates optimal performance

### Key Design Choices

| Choice | Reason |
|--------|--------|
| Input Features | Correlator (main), h_estimate, SNR, energy → interpretable + robust |
| 3 Hidden Layers | Balance model capacity (avoid overfitting on 100k samples) |
| BatchNorm | Stabilize training, reduce internal covariate shift |
| Dropout | Regularization to prevent overfitting |
| Class Weights | Handle any label imbalance (50/50 target) |

### Expected Performance

- **Target FNR**: ≤ 10⁻⁷ (equal to best Monte Carlo seen in project)
- **Target FPR**: ≤ 10⁻⁷ (maintain false alarm probability)
- **Advantage**: No threshold tuning needed; NN learns adaptive decision boundary

### References

**[1]** Braca, P., Millefiori, L. M., Aubry, A., Marano, S., De Maio, A., & Willett, P. (2022). "Statistical Hypothesis Testing Based on Machine Learning: Large Deviations Analysis." *IEEE Open Journal of Signal Processing*, 3, 464-495. https://doi.org/10.1109/OJSP.2022.3232284

**[2]** Neyman, J., & Pearson, E. S. (1933). "On the Problem of the Most Efficient Tests of Statistical Hypotheses." *Philosophical Transactions of the Royal Society A*, 231, 289-337.

**[3]** Dembo, A., & Zeitouni, O. (1998). *Large Deviations Techniques and Applications*. 2nd ed., Springer.